In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ON_KAGGLE = Path("/kaggle").exists()

if ON_KAGGLE:
    # Repo in /tmp, così non finisce negli output
    repo = Path("/tmp/repo")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Matteo22T/Deep-Learning-Project.git", str(repo)],
            check=True,
        )
    sys.path.append(str(repo))

    found = list(Path("/kaggle/input").rglob("synthetic_with_labels.csv"))
    assert found, "synthetic_with_labels.csv non trovato in /kaggle/input"
    data_dir = found[0].parent

    # save_fig scrive in results/figures relativo alla cwd
    os.chdir("/kaggle/working")
else:
    current_dir = Path.cwd()
    sys.path.append(str(current_dir.parent if current_dir.name == "notebooks" else current_dir))
    data_dir = Path("../data") if Path("../data").exists() else Path("data")


In [ ]:
from src.data.loader import extract_real_patient_id, load_all_datasets
from src.visualization.plots import (
    plot_cosine_similarity_and_dominance,
    plot_latent_space_projections,
    plot_target_distribution,
)

df_synth, df_real_in, df_real_out, df_gt = load_all_datasets(data_dir)

# 1. Tabella di contingenza 2x2 sui 50 pazienti reali unici e controllo varianza feature
df_real_unique = (
    df_real_in.assign(base_id=df_real_in["patient_id"].apply(extract_real_patient_id))
    .groupby("base_id")
    .first()
)

print("=== Tabella di Contingenza 2x2: Coorte Interna (N=50 Pazienti Unici) ===")
ct_internal = pd.crosstab(
    df_real_unique["OS_1y"],
    df_real_unique["PFS_1y"],
    margins=True,
    rownames=["OS_1y"],
    colnames=["PFS_1y"],
)
display(ct_internal)

feat_cols = [c for c in df_real_in.columns if c.startswith("f")]
min_std = df_real_unique[feat_cols].std().min()
print(f"✓ Minima deviazione standard tra le 768 feature (N=50 reali): {min_std:.6f} (> 0, zero feature costanti)\n")

# 2. Generazione Figura 1 (Corretta)
fig1 = plot_target_distribution(df_real_in, df_synth, save=True)
plt.show()

# 3. Generazione Figura 2 (Cosine Similarity & Dominanza Host A vs Donor B)
fig2, df_sim_summary = plot_cosine_similarity_and_dominance(df_real_in, df_synth, save=True)
display(df_sim_summary.round(4))
plt.show()

# 4. Generazione Figura 3 (PCA & UMAP/t-SNE Space Geometry & Domain Shift)
fig3 = plot_latent_space_projections(df_real_in, df_synth, df_real_out, df_gt=df_gt, save=True)
plt.show()